# JobBERT: job description → structured fields → resume match %

**Goal:** read a job post into structured fields (skills, years, degree, work mode), read the resume the same way, then score the match. Then check whether it beats Hiriq's current rules reader.

| Step | What | Model |
|---|---|---|
| 1 | Settings | – |
| 2 | Load a job post (saved Google page, scraper JSON or pasted text) | – |
| 3 | Clean the text and split it into sections and sentences | rules |
| 4 | Load JobBERT (2 models) and MiniLM | `jjzha/jobbert_knowledge_extraction` (hard skills: Python, Power BI), `jjzha/jobbert_skill_extraction` (soft and applied skills: "data storytelling") |
| 5 | Tag every sentence and merge B/I tags into spans | JobBERT |
| 6 | Filter spans into **tools**, **domain** and **soft** skills; drop degree words and fragments; merge near-duplicates | rules + Hiriq skill names + MiniLM |
| 7 | The other fields: years, degree, work mode, type | rules |
| 8 | Read the resume the same way | JobBERT + rules |
| 9 | Semantic match for skills the rules miss | `all-MiniLM-L6-v2` |
| 10 | Match % with the explained breakdown (Hiriq's scorer) | – |
| 11 | JobBERT vs Hiriq's current reader, side by side | – |
| 12 | Every scraped job ranked | – |
| 13 | Accuracy against your hand labels (precision / recall) | – |

**One rule from Hiriq stays:** a skill counts only with evidence in the text. A semantic (embedding) match can lift a missing skill to *related* (half credit), never to a full match, and only with a resume line that really contains it.

**Setup, once (about 1 GB download the first time, cached after that):**
```
pip install torch transformers sentence-transformers pandas beautifulsoup4 pdfplumber python-docx
```
Run it from `ml/notebooks/`: it imports Hiriq's own skill names and scorer from `../../backend`.

## Step 1 · Settings

In [79]:
JD_SOURCE = "sample"        # "sample" = the saved Google page; or a path to gjobs_*.json from the scraper notebook; or "text"
JD_INDEX = 1                # which job from that source (0 = first)
JD_TEXT = ""                # used when JD_SOURCE == "text": paste a job description here
JD_TITLE = ""

RESUME_PATH = r"C:\Users\Zahid Shaikh\z\Hiriq\Zahid_Salim_Shaikh__Copy_.pdf"   # your resume: .pdf, .docx or .txt
RESUME_YEARS = None         # None = work it out from the job dates in the resume; or a number like 4.5

NER_MIN_SCORE = 0.50        # a span naming a skill Hiriq already knows needs this confidence
NEW_MIN_SCORE = 0.80        # a name Hiriq doesn't know yet (MLflow, Kubeflow) needs this much more
DUP_SIM = 0.80              # two job skills this similar are one skill ("ml/dl models" ≈ "ml/dl algorithms")
MAX_TOOLS = 25              # caps, so a long post can't bury the score under dozens of items
MAX_DOMAIN = 8
RELATED_SIM = 0.62          # embedding similarity for "related" (half credit)
SOFT_SIM = 0.55             # a soft skill counts as covered when a resume line is this similar
DEVICE = None               # None = GPU if there is one, else CPU
DRY_RUN = False             # True = no models (a stand-in tagger), to check the plumbing quickly

## Step 2 · Imports, Hiriq's helpers, and the job post

In [80]:
import json, os, re, sys
from collections import defaultdict
from datetime import date
from pathlib import Path

import pandas as pd

BACKEND = Path("../../backend").resolve()
sys.path.insert(0, str(BACKEND))
from app.services import matching, postingread                      # Hiriq's skill names (aliases) and scorer
from app.services.resume import parse_month                          # "Apr 2024" -> month index
from app.services.sources import google_jobs_parse as gp             # reading a saved Google Jobs page

import app.services.resume_text as _rt
print("backend:", BACKEND, "| PDF fix loaded:", hasattr(_rt, "looks_glued"))
if not hasattr(_rt, "looks_glued"):
    print("⚠ This backend has no PDF fix. Extract hiriq-jobbert-v2.zip's backend/ files into", BACKEND, "and restart the kernel.")

CANDIDATE_WEIGHTS = {"skills": 42, "qualification": 21, "experience": 32, "location": 0, "keywords": 5}   # same as the app

pd.set_option("display.max_colwidth", 120)


def _header(text, label):
    m = re.search(rf"^\s*{label}\s*:\s*(.+)$", text, re.I | re.M)
    return m.group(1).strip() if m else None


def load_jobs(source):
    if source == "sample":
        return gp.parse_page((BACKEND / "tests/fixtures/google_jobs_sample.html").read_text(encoding="utf-8"))
    if source == "text":
        t = JD_TEXT
        desc = re.sub(r"^\s*(?:[a-z]\)|\d+[.)])\s+", "", t, flags=re.M | re.I)      # drop "a)", "1)", "2." numbering
        return [{"title": JD_TITLE or _header(t, "(?:job )?title") or "Pasted job", "company": _header(t, "company"),
                 "location": _header(t, r"(?:job\s+)?location"), "employment_type": _header(t, "employment type"), "description": desc}]
    return json.loads(Path(source).read_text(encoding="utf-8"))


JOBS = load_jobs(JD_SOURCE)
JOB = JOBS[min(JD_INDEX, len(JOBS) - 1)]
print(f"{len(JOBS)} job(s) in the source. Using #{JD_INDEX}: {JOB['title']} · {JOB.get('company')} · {JOB.get('location')}")
print(JOB["description"][:500], "...")

backend: C:\Users\Zahid Shaikh\z\Hiriq\hiriq\hiriq\backend | PDF fix loaded: False
⚠ This backend has no PDF fix. Extract hiriq-jobbert-v2.zip's backend/ files into C:\Users\Zahid Shaikh\z\Hiriq\hiriq\hiriq\backend and restart the kernel.
2 job(s) in the source. Using #1: Senior Analyst, Digital Analytics · Labcorp · Bengaluru, Karnataka
Labcorp is seeking a Senior Analyst – Digital Analytics to join our team at LabCorp Bangalore
Work Schedule
The Senior Associate, Website and Campaign Performance Reporting will be responsible for delivering website, campaign, and marketing performance reporting and insights using Adobe Customer Journey Analytics (CJA) and external marketing data sources. The role will focus on data analysis, reporting automation, data storytelling, and presenting actionable insights to marketing stakeholders.
J ...


## Step 3 · Clean the text, find the sections, split into sentences

Google often runs headings into the next sentence ("About the Role**We** are…", "ecosystem.**Key Responsibilities**Data…"). We split those back apart, but never inside a real word like PowerPoint or JavaScript (checked against Hiriq's skill names).

Sections decide whether a skill is **required** or **nice to have**, and stop us reading skills from "About the company" or the legal footer.

| Section | Heading examples | Skills count as |
|---|---|---|
| `responsibilities` | Key Responsibilities, What you'll do | required |
| `requirements` | Minimum Qualifications, Requirements, Core Technical Skills | required |
| `preferred` | Preferred Qualifications, Nice to have, Good to have, Bonus | nice |
| `soft` | Soft Skills & Mindset | soft skills |
| `about` / `benefits` / `legal` | About us, Benefits, Equal Opportunity | ignored |
| `body` | anything before the first heading | required |

In [81]:
SECTION_WORDS = [
    ("preferred", r"preferred|nice[\s-]to[\s-]have|good[\s-]to[\s-]have|bonus|desirable|plus points?"),
    ("soft", r"soft skills?|mindset|personal (?:attributes|qualities)"),
    ("requirements", r"requirements?|qualifications?|what we['’]re looking for|who you are|must have|technical skills|core skills|skills|eligibility|experience required"),
    ("responsibilities", r"responsibilit(?:y|ies)|what you['’]ll do|your role|the role|key tasks|duties|job description"),
    ("benefits", r"benefits|perks|what we offer|why join"),
    ("legal", r"equal opportunity|eeo|accommodation|privacy|disclaimer|how to apply"),
    ("about", r"about (?:us|the company|the team)|who we are|company overview"),
]
HEADINGS = [
    "About the Role", "About the Job", "About Us", "About the Company", "Key Responsibilities", "Job Responsibilities", "Responsibilities",
    "Candidate Requirements", "Requirements", "Core Technical Skills", "Technical Skills", "Minimum Qualifications", "Preferred Qualifications",
    "Qualifications", "Soft Skills & Mindset", "Soft Skills", "Nice to Have", "Good to Have", "What You'll Do", "What We're Looking For",
    "Who You Are", "Benefits", "Work Schedule", "Job Description",
]
CAMEL_OK = {"powerpoint", "javascript", "typescript", "linkedin", "github", "gitlab", "youtube", "powershell", "bigquery", "pyspark",
            "numpy", "scipy", "mongodb", "postgresql", "mysql", "nosql", "openai", "hubspot", "wordpress", "sharepoint", "quickbooks",
            "salesforce", "databricks", "devops", "mlops", "fastapi", "pytorch", "tensorflow", "iphone", "ebay", "paypal", "mcdonald"}


def _keep_camel(word):
    k = re.sub(r"[^a-z0-9]", "", word.lower())
    return k in CAMEL_OK or matching._key(word) in matching.ALIAS_KEY


def clean_text(text):
    t = (text or "").replace("\r", "\n").replace("\xa0", " ")
    t = re.sub(r"[•▪●◦]\s*", "\n• ", t)
    for h in sorted(HEADINGS, key=len, reverse=True):          # "Key ResponsibilitiesData" -> "Key Responsibilities\nData"
        t = re.sub(rf"({re.escape(h)})(?=[A-Z])", r"\n\1\n", t, flags=re.I)
    t = re.sub(r"(?<=[a-z0-9)%])\.(?=[A-Z])", ".\n", t)          # "ecosystem.Key" -> "ecosystem.\nKey"
    # any other lower→Upper join ("MindsetAnalytical"), unless the word is a real one (PowerPoint, JavaScript) or the left part
    # is short, as in names (LabCorp, YouTube, HubSpot)
    t = re.sub(r"\b([A-Za-z]*[a-z])([A-Z][a-z]+)\b",
               lambda m: m.group(0) if len(m.group(1)) < 5 or _keep_camel(m.group(0)) else f"{m.group(1)}\n{m.group(2)}", t)
    return "\n".join(l.strip() for l in t.splitlines() if l.strip())


def heading_kind(line):
    """A line is a heading when it is short, has no full stop, and names a section."""
    s = line.strip(" :•-").strip()
    if not s or len(s.split()) > 6 or s.endswith(".") or ":" in s[:-1]:
        return None
    for kind, pat in SECTION_WORDS:
        if re.search(rf"\b(?:{pat})\b", s, re.I):
            return kind
    return None


def sentences(text):
    """-> [{"text", "section"}] in order."""
    out, section = [], "body"
    for line in clean_text(text).splitlines():
        kind = heading_kind(line)
        if kind:
            section = kind
            continue
        for s in re.split(r"(?<=[.!?])\s+(?=[A-Z(])", line.lstrip("• ").strip()):
            # an inline label ("ETL & Data Governance: Manage basic ETL…") is a heading, not a product name: it goes in
            # lower case on its own, so its capitals don't make "Data Governance" look like a tool
            m = re.match(r"^([A-Z][^:.]{2,60}):\s+(\S.*)$", s.strip())
            if m and len(m.group(1).split()) <= 6:
                label, s = m.group(1), m.group(2)
                if len(label.split()) >= 2:
                    out.append({"text": label.lower(), "section": section})
            if len(s.split()) >= 2:
                out.append({"text": s.strip(), "section": section})
    return out


SENTS = sentences(JOB["description"])
print(len(SENTS), "sentences")
display(pd.DataFrame(SENTS).head(25))

37 sentences


,text,section
0,Labcorp is seeking a Senior Analyst – Digital Analytics to join our team at LabCorp Bangalore,body
1,Work Schedule,body
2,"The Senior Associate, Website and Campaign Performance Reporting will be responsible for delivering website, campaig...",body
3,"The role will focus on data analysis, reporting automation, data storytelling, and presenting actionable insights to...",body
4,Own and deliver website performance reporting using Adobe Customer Journey Analytics (CJA).,responsibilities
5,"Analyze website traffic, user behavior, engagement, conversion trends, and customer journeys and translate data into...",responsibilities
6,"Develop and maintain campaign and marketing performance reporting, including channel, campaign, and marketing perfor...",responsibilities
7,"Be well versed in reporting and analyzing campaign and marketing data imported from external data sources, such as M...",responsibilities
8,"Build and maintain CJA segments, calculated/custom fields, metrics, dimensions, and reporting components to support ...",responsibilities
9,"Automate CJA reporting and dashboards using Power BI, reducing manual reporting effort and improving reporting effic...",responsibilities


## Step 4 · Load JobBERT

Both models are BERT models fine-tuned on job posts (the SkillSpan paper). They tag each word **B** (a span begins), **I** (it continues) or **O** (outside).

- **knowledge** model: hard skills, tools, subjects (*Python*, *Adobe Customer Journey Analytics*, *ETL*)
- **skill** model: soft and applied skills (*data storytelling*, *manage multiple reporting requirements*)

The first run downloads about 430 MB per JobBERT model and 90 MB for MiniLM (sentence similarity).

In [82]:
def load_taggers():
    if DRY_RUN:
        return None, None
    import torch
    from transformers import pipeline

    dev = DEVICE if DEVICE is not None else (0 if torch.cuda.is_available() else -1)
    kw = dict(task="token-classification", aggregation_strategy="first", device=dev)
    return (pipeline(model="jjzha/jobbert_knowledge_extraction", **kw),
            pipeline(model="jjzha/jobbert_skill_extraction", **kw))


def load_embedder():
    if DRY_RUN:
        return None
    from sentence_transformers import SentenceTransformer

    return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


KNOWLEDGE_TAGGER, SKILL_TAGGER = load_taggers()
EMBEDDER = load_embedder()                 # used in step 6 (near-duplicates) and step 9 (semantic match)
print("taggers:", "stand-in (DRY_RUN)" if DRY_RUN else "JobBERT knowledge + skill", "· embedder:", "word overlap" if EMBEDDER is None else "MiniLM")


def similarity(a_list, b_list):
    """Cosine similarity matrix (len(a) × len(b))."""
    if not a_list or not b_list:
        return [[0.0] * len(b_list) for _ in a_list]
    if EMBEDDER is None:                                    # stand-in: word overlap
        def words(s):
            return set(re.findall(r"[a-z0-9]+", s.lower()))
        return [[len(words(a) & words(b)) / max(1, len(words(a) | words(b))) for b in b_list] for a in a_list]
    va = EMBEDDER.encode(a_list, normalize_embeddings=True, convert_to_numpy=True)
    vb = EMBEDDER.encode(b_list, normalize_embeddings=True, convert_to_numpy=True)
    return (va @ vb.T).tolist()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

taggers: JobBERT knowledge + skill · embedder: MiniLM


## Step 5 · Tag every sentence, then merge B/I into spans

These models use plain `B`/`I`/`O` labels (not `B-SKILL`), so the Hugging Face pipeline returns **B** pieces and **I** pieces as separate groups. We join an **I** onto the span before it when only spaces sit between them:

| Pipeline output | Merged span |
|---|---|
| `B "Adobe"` `I "Customer Journey Analytics"` | **Adobe Customer Journey Analytics** |
| `B "Power BI"` | **Power BI** |
| `I "storytelling"` (no B before) | **storytelling** (an orphan I starts its own span) |

In [83]:
def merge_bio(pieces, text):
    spans = []
    for p in sorted(pieces, key=lambda x: x["start"]):
        tag = p.get("entity_group") or p.get("entity", "")
        if tag == "O":
            continue
        if spans and tag.endswith("I") and text[spans[-1]["end"]:p["start"]].strip() == "":
            s = spans[-1]
            s["end"], s["scores"] = p["end"], s["scores"] + [float(p["score"])]
        else:
            spans.append({"start": p["start"], "end": p["end"], "scores": [float(p["score"])]})
    for s in spans:
        # JobBERT sometimes stops inside a product name ("Adobe" | "Customer Journey Analytics", "Google Apps" | "Script"):
        # a span ending in a capitalised word takes the capitalised words right after it (at most 5 words in all)
        while text[s["start"]:s["end"]].split() and text[s["start"]:s["end"]].split()[-1][:1].isupper():
            m = re.match(r"\s([A-Z][A-Za-z0-9+#]*)\b(?!\s*:)", text[s["end"]:])     # never into a "Label:" word
            if not m or len(text[s["start"]:s["end"]].split()) >= 5:
                break
            s["end"] += m.end()
        s["text"] = text[s["start"]:s["end"]].strip(" ,.;:()-–")
        s["score"] = round(sum(s["scores"]) / len(s["scores"]), 3)
        del s["scores"]
    # a span now inside a longer one ("Journey Analytics" inside "Adobe Customer Journey Analytics") is the same mention
    spans = [s for s in spans if s["text"] and not any(o is not s and o["start"] <= s["start"] and s["end"] <= o["end"]
                                                        and (o["end"] - o["start"]) > (s["end"] - s["start"]) for o in spans)]
    return spans


# --- the stand-in tagger for DRY_RUN: Hiriq's skill names as "knowledge", a few phrases as "skill" ---
_SOFT_STANDIN = re.compile(r"\b(data storytelling|communication skills?|stakeholder management|problem[- ]solving|attention to detail|"
                           r"presentation skills|analytical (?:thinking|skills)|work independently)\b", re.I)


def _standin(text, kind):
    pats = [p for _, p in postingread._vocab()] if kind == "knowledge" else [_SOFT_STANDIN]
    return [{"entity_group": "B", "score": 0.9, "start": m.start(), "end": m.end()} for p in pats for m in p.finditer(text)]


def tag(texts, kind, batch_size=16):
    """-> one list of merged spans per input text."""
    tagger = KNOWLEDGE_TAGGER if kind == "knowledge" else SKILL_TAGGER
    raw = [_standin(t, kind) for t in texts] if tagger is None else tagger(texts, batch_size=batch_size)
    return [merge_bio(r, t) for r, t in zip(raw, texts)]


def tag_sentences(sents):
    texts = [s["text"][:1500] for s in sents]          # a sentence never gets near BERT's 512-token limit at this length
    rows = []
    for kind in ("knowledge", "skill"):
        for i, spans in enumerate(tag(texts, kind)):
            for sp in spans:
                if sp["score"] >= NER_MIN_SCORE:
                    rows.append({"kind": kind, "span": sp["text"], "score": sp["score"], "section": sents[i]["section"], "sentence": sents[i]["text"]})
    return rows


SPANS = tag_sentences(SENTS)
print(len(SPANS), "spans")
display(pd.DataFrame(SPANS)[["kind", "span", "score", "section"]])

80 spans


,kind,span,score,section
0,knowledge,Adobe Customer Journey Analytics,0.991,body
1,knowledge,CJA,0.651,body
2,knowledge,data analysis,1.000,body
3,knowledge,reporting automation,0.985,body
4,knowledge,data storytelling,0.997,body
...,...,...,...,...
75,skill,manage multiple reporting requirements,1.000,requirements
76,skill,prioritize competing requests,1.000,requirements
77,skill,deliver high-quality outputs within timelines,0.998,requirements
78,skill,business mindset,1.000,requirements


## Step 6 · From spans to skills (filtered)

The first run showed JobBERT returns **too much**: 51 "required skills" on the HDFC post, including degree words, fragments and near-duplicates. Every extra item counts against the candidate, so the score sank. Each knowledge span now goes through these rules:

| Span | Rule | Goes to |
|---|---|---|
| "SQL, Python, and PySpark" | split lists on `,` `/` `and` `or` | 3 pieces, each checked below |
| "Python programming", "Automate CJA reporting using Power BI" | contains a skill Hiriq knows (confidence ≥ `NER_MIN_SCORE`) | **tool**: python / power bi |
| "MLflow", "Kubeflow", "Adobe CJA", "AWS" | not known, but looks like a product name (capitals, acronym, camel case), confidence ≥ `NEW_MIN_SCORE` | **tool** (new name) |
| "fraud detection", "feature stores", "big data pipelines" | 2–4 lowercase words, a real subject, confidence ≥ `NEW_MIN_SCORE` | **domain**: nice-to-have, matched by meaning only |
| "Bachelor's", "Master's in a", "Computer Science" (in a degree sentence) | degree words, or a subject in the sentence naming a degree | dropped (the degree is read in step 7) |
| "banking", "cross", "service", "logs", "your" | one lowercase word, or generic | dropped |
| "ml/dl models" ≈ "ml/dl algorithms" ≈ "ml/dl" | similarity ≥ `DUP_SIM`, or one contains the other plus only filler words | kept once |
| "Adobe" next to "Adobe Journey Optimizer" | the longer one adds real words, so it is a different product | both kept |
| "Model Context Protocol (MCP", "CJA", "Adobe CJA" | "(…)" removed; an acronym defined in the post ("… (CJA)") means the full name | Model Context Protocol · Adobe Customer Journey Analytics |
| "Adobe" + "Customer Journey Analytics", "Google Apps" + "Script" | a span ending in a capitalised word takes the capitalised words right after it | one product name |
| "AI", "BI", "IT" alone | too vague to check in a resume | dropped |

**Soft skills** keep only short phrases (≤ 4 words) that name a quality, not a task: "data storytelling", "attention to detail", "communication". Task phrases like "Design, develop, validate and deploy ML/DL models" are dropped. Soft skills are shown but stay out of the %.

Then: anything in a `preferred` section is **nice**, anything else **required**. A tool seen as required anywhere stays required.

In [84]:
GENERIC = {"data", "datasets", "complex datasets", "large datasets", "reports", "reporting", "insights", "dashboards", "tools", "platforms",
           "metrics", "dimensions", "systems", "solutions", "workflows", "business", "analysis", "analytics", "technology", "technologies",
           "experience", "skills", "knowledge", "requirements", "teams", "stakeholders", "sources", "data sources", "external data sources",
           "marketing platforms", "other marketing platforms", "trends", "performance", "process", "processes", "automation", "frameworks",
           "standards", "stores", "repositories", "logs", "use cases", "applications", "models", "pipelines", "projects", "products",
           "services", "service", "infrastructure", "operations", "environment", "methodologies", "techniques", "concepts", "domain"}
STOP = {"a", "an", "the", "of", "in", "on", "for", "to", "with", "and", "or", "your", "our", "their", "its", "this", "that", "all", "any",
        "such", "as", "via", "by", "at", "from", "into", "across", "within", "using", "including", "e.g", "etc"}
SOFT_WORDS = {"communication", "leadership", "teamwork", "collaboration", "ownership", "curiosity", "passion", "creativity", "adaptability",
              "accountability", "integrity", "mentoring", "negotiation", "storytelling", "presentation", "empathy", "initiative", "resilience",
              "analytical", "organized", "organised", "detail-oriented", "problem-solving", "prioritization", "prioritisation", "coaching",
              "influencing", "persuasion", "curious", "proactive"}
TASK_VERBS = re.compile(r"^(design|develop|build|create|deploy|process|research|conduct|ensure|maintain|optimi[sz]e|present|collaborate|drive|"
                        r"driving|take|manage|lead|own|deliver|identify|analy[sz]e|implement|support|work|uncover|solve|explore|exploring|"
                        r"building|designing|processing|addressing|partner|monitor|define|apply|foster|prototype)\b", re.I)
DEGREE_WORDS = re.compile(r"\b(bachelor|master['’]?s|masters?\b|degree|diploma|ph\.?\s?d|doctorate|graduat|b\.?\s?tech|m\.?\s?tech|m\.?\s?sc|mba)", re.I)
SUBJECTS = {"statistics", "mathematics", "maths", "economics", "computer science", "data science", "engineering", "physics", "finance",
            "operations research", "information technology", "commerce", "business administration"}
# short forms JobBERT returns that Hiriq's skill list doesn't map (yet): one name each, or two skills for "ML/DL"
EXPAND = {"genai": ["generative ai"], "gen ai": ["generative ai"], "generative ai": ["generative ai"], "ml/dl": ["machine learning", "deep learning"],
          "ml": ["machine learning"], "dl": ["deep learning"], "ai/ml": ["machine learning"], "ml/ai": ["machine learning"],
          "large language models": ["llm"], "large language model": ["llm"], "llms": ["llm"], "eda": ["exploratory data analysis"],
          "exploratory data analysis": ["exploratory data analysis"], "mlops": ["mlops"]}
# "/" splits "SQL / Google SQL" but not "A/B testing", "CI/CD" or "PL/SQL" (both sides need 3+ characters)
_SPLIT = re.compile(r"\s*(?:,|;|\band\b|\bor\b|&)\s*|(?<=[A-Za-z0-9]{3})\s*/\s*(?=[A-Za-z0-9]{3})", re.I)
_EDGE = re.compile(r"^(?:the|a|an|of|in|with|using|via|such as|like|including|strong|solid|good|hands-on|basic|advanced|demonstrable|"
                   r"experience (?:in|with)|knowledge of|proficiency in|familiarity with)\s+|\s+(?:skills?|tools?|experience|programming|querying|"
                   r"platforms?|frameworks?|libraries|solutions)$", re.I)


def known_inside(phrase):
    """Hiriq skill names found inside a phrase (plus the EXPAND short forms)."""
    low = phrase.lower().strip()
    if low in EXPAND:
        return EXPAND[low]
    found = [c for c, pat in postingread._vocab() if pat.search(phrase)]
    for short, names in EXPAND.items():           # "ML/DL models" contains "ml/dl"
        if re.search(rf"(?<![a-z0-9]){re.escape(short)}(?![a-z0-9])", low):
            found += [n for n in names if n not in found]
    return found


_ACRO_DEF = re.compile(r"((?:[A-Z][A-Za-z0-9+#.]*\s+){1,5}[A-Z][A-Za-z0-9+#.]*)\s*\(([A-Z][A-Za-z0-9&]{1,9})\)")
VAGUE = {"ai", "bi", "it", "ml", "data", "cloud"}       # alone these say nothing; "Power BI", "AI agents", "ML" via EXPAND still count


def acronyms(sents):
    """'Adobe Customer Journey Analytics (CJA)' -> {"cja": "Adobe Customer Journey Analytics"}, so a later "CJA" or
    "Adobe CJA" is the same skill as the full name."""
    out = {}
    for s in sents:
        for m in _ACRO_DEF.finditer(s["text"]):
            out.setdefault(m.group(2).lower(), m.group(1).strip())
    return out


def pieces(span):
    """A span split into list items, with filler words trimmed from the edges and "(…)" removed."""
    out = []
    span = re.sub(r"\([^)]*\)?|\)", " ", span or "")          # "Model Context Protocol (MCP" -> "Model Context Protocol"
    for part in _SPLIT.split(span):
        part = part.strip(" .()[]'\"-–:")
        for _ in range(3):                       # "strong hands-on experience with X" needs several passes
            part = _EDGE.sub("", part).strip()
        words = part.split()
        while words and words[-1].lower() in STOP:
            words.pop()
        while words and words[0].lower() in STOP:
            words.pop(0)
        if words:
            out.append(" ".join(words))
    return out


def tool_like(part):
    """Looks like a product or acronym: MLflow, Kubeflow, AWS, Adobe CJA, Jira, Adobe Customer Journey Analytics. Every word
    starts with a capital or digit, or is camel case / an acronym / has symbols (C#, .NET). "IT service desk" does not."""
    words = part.split()
    if not 1 <= len(words) <= 4:
        return False
    return all(w[0].isupper() or w[0].isdigit() or re.search(r"[a-z][A-Z]|[#+.]\w", w) for w in words)


def classify(part, score, in_degree_sentence=False):
    """-> "known", "tool", "domain" or None (dropped)."""
    low = part.lower()
    if not part or len(part) < 2 or low in GENERIC or low in STOP or (low in VAGUE and low not in EXPAND):
        return None
    if DEGREE_WORDS.search(part):
        return None
    if in_degree_sentence and low in SUBJECTS:   # "Computer Science, Statistics, Mathematics" after "degree in"
        return None
    if low in EXPAND:
        return "known" if score >= NER_MIN_SCORE else None
    if known_inside(part):
        return "known" if score >= NER_MIN_SCORE else None
    if in_degree_sentence:
        return None
    if score < NEW_MIN_SCORE:
        return None
    if tool_like(part) and not (len(part.split()) == 1 and part.istitle() and len(part) > 12):
        return "tool"
    words = [w for w in re.findall(r"[a-z][a-z\-]+", low) if w not in STOP]
    if 2 <= len(part.split()) <= 4 and len(words) >= 2 and not all(w in GENERIC for w in words):
        return "domain"
    return None


SOFT_PHRASES = re.compile(r"\b(attention to detail|stakeholder management|time management|critical thinking|problem[- ]solving|"
                          r"business (?:mindset|acumen)|work ethic|growth mindset|decision[- ]making|self[- ]starter|team player)\b", re.I)
SOFT_TAIL = re.compile(r"\b(skills?|mindset|thinking|attitude|acumen|abilit(?:y|ies))$", re.I)


def soft_ok(span):
    """A soft skill names a quality ("data storytelling", "presentation skills", "attention to detail"), not a task
    ("reducing manual reporting effort", "prioritize competing requests")."""
    words = span.split()
    low = span.lower().strip()
    if not 1 <= len(words) <= 4 or words[0].lower() in STOP or TASK_VERBS.match(low) or low in GENERIC:
        return False
    if SOFT_PHRASES.search(low):
        return True
    tokens = set(re.findall(r"[a-z][a-z\-]+", low))
    if tokens & SOFT_WORDS:
        return True
    return len(words) >= 2 and bool(SOFT_TAIL.search(low)) and not words[0].lower().endswith("ing")


FILLER = GENERIC | {"models", "model", "algorithms", "pipeline", "development", "creation", "platform", "framework", "architectures",
                    "architecture", "integration", "integrations", "api", "apis", "library", "techniques", "concepts", "capabilities"}


def _contains_with_filler(short, long):
    """'ml/dl' inside 'ml/dl models' (only filler added) is a duplicate; 'adobe' inside 'adobe journey optimizer' is not."""
    if not re.search(rf"(?<![a-z0-9]){re.escape(short)}(?![a-z0-9])", long):
        return False
    extra = re.sub(rf"(?<![a-z0-9]){re.escape(short)}(?![a-z0-9])", " ", long).split()
    return all(w in FILLER or w in STOP for w in extra)


def dedupe(items):
    """Keep one of near-duplicates: known names first, then the shorter one."""
    items = sorted(items, key=lambda x: (x["from"] != "known", len(x["skill"])))
    names = [x["skill"] for x in items]
    sims = similarity(names, names)
    keep = []
    for i, it in enumerate(items):
        twin = next((k for k in keep if sims[i][k] >= DUP_SIM or _contains_with_filler(names[k], names[i])), None)
        if twin is None:
            keep.append(i)
        else:
            it["dup_of"] = names[twin]
            if it["need"] == "required":
                items[twin]["need"] = "required"
    return [items[i] for i in keep], [it for it in items if "dup_of" in it]


def to_skills(spans, sents=None):
    found, soft, discovered, dropped = {}, {}, [], []
    acro = acronyms(sents if sents is not None else [{"text": r["sentence"]} for r in spans])

    def add(name, row, source, group):
        low = name.lower()
        words = low.split()
        if low in acro:                                   # "CJA" -> "Adobe Customer Journey Analytics"
            name, group = acro[low], "tool"
        elif len(words) == 2 and words[1] in acro and acro[words[1]].lower().startswith(words[0]):   # "Adobe CJA"
            name, group = acro[words[1]], "tool"
        key = matching.canon(name)
        need = "nice" if row["section"] == "preferred" else "required"
        cur = found.get(key)
        if cur is None:
            found[key] = {"skill": key, "as_written": name, "group": group, "need": need, "score": row["score"], "from": source, "evidence": row["sentence"]}
        elif need == "required":
            cur["need"] = "required"

    for row in spans:
        if row["section"] in ("about", "benefits", "legal"):
            continue
        span = row["span"]
        degree_sentence = bool(matching.degree_levels(row["sentence"]))
        if row["kind"] == "skill" or row["section"] == "soft":
            if soft_ok(span) and row["score"] >= NER_MIN_SCORE:
                soft.setdefault(span.lower(), {"soft_skill": span, "score": row["score"], "evidence": row["sentence"]})
            for k in known_inside(span):         # a soft span can still name a tool ("present using PowerPoint")
                add(k, row, "known", "tool")
            continue
        for part in pieces(span):
            kind = classify(part, row["score"], degree_sentence)
            if kind == "known":
                for k in known_inside(part):
                    add(k, row, "known", "tool")
            elif kind == "tool":
                add(part, row, "new", "tool")
            elif kind == "domain":
                add(part, row, "new", "domain")
            else:
                dropped.append(part)
                if len(part.split()) > 4:
                    discovered.append(part)
    tools, tool_dups = dedupe([x for x in found.values() if x["group"] == "tool"])
    domain, dom_dups = dedupe([x for x in found.values() if x["group"] == "domain"])
    tool_names = [t["skill"] for t in tools]
    domain = [d for d in domain if not any(_contains_with_filler(t, d["skill"]) for t in tool_names)]
    tools.sort(key=lambda x: (x["need"] != "required", x["from"] != "known"))
    skills = tools[:MAX_TOOLS] + domain[:MAX_DOMAIN]
    return skills, list(soft.values()), discovered, {"dropped": dropped, "merged": tool_dups + dom_dups}


SKILLS, SOFT, DISCOVERED, AUDIT = to_skills(SPANS, SENTS)
display(pd.DataFrame(SKILLS)[["skill", "as_written", "group", "need", "from", "score"]])
print("soft skills:", [s["soft_skill"] for s in SOFT])
print("merged as duplicates:", [(d["skill"], "→", d["dup_of"]) for d in AUDIT["merged"]])
print("dropped:", sorted(set(AUDIT["dropped"]))[:40])

,skill,as_written,group,need,from,score
0,power bi,power bi,tool,required,known,0.998
1,marketo,Marketo,tool,required,new,0.988
2,powerpoint,PowerPoint,tool,required,new,1.000
3,adobe journey optimizer,Adobe Journey Optimizer,tool,required,new,0.987
4,adobe customer journey analytics,Adobe Customer Journey Analytics,tool,required,new,0.991
5,website traffic,website traffic,domain,required,new,0.997
6,data storytelling,data storytelling,domain,required,new,0.997
7,website analytics,website analytics,domain,required,new,1.000
8,digital marketing,digital marketing,domain,required,new,1.000
9,customer journeys,customer journeys,domain,required,new,0.977


soft skills: ['data storytelling skills', 'analytical and problem-solving skills', 'data storytelling', 'presentation skills', 'business mindset']
merged as duplicates: []
dropped: ['CJA', 'analysis', 'automation', 'calculated/', 'campaigns', 'channels', 'conversion', 'data', 'data analysis', 'engagement', 'fields', 'reporting', 'reporting automation', 'segmentation']


## Step 7 · The other fields, and the structured job

JobBERT only tags skills. Years, degree, work mode and job type come from rules:

| Field | How | Example |
|---|---|---|
| experience_min / max | a number or range next to the word "experience" (Hiriq's reader) | "5 to 8+ years of professional experience" → 5, 8 |
| qualification | the sentence naming a real degree, and its level | "Bachelor's in Computer Science" → bachelor's |
| work_mode | remote / hybrid / onsite words | "Hybrid (3 days in office)" → hybrid |
| employment_type | from Google's chip, else the text | "Full–time" |

In [85]:
_DEGREE_LINE = re.compile(r"\b(degree|bachelor|master|b\.?\s?tech|m\.?\s?tech|b\.?\s?e\b|mba|phd|graduat)", re.I)


def work_mode(text):
    t = text.lower()
    if re.search(r"\bhybrid\b", t):
        return "hybrid"
    if re.search(r"\b(fully remote|remote[- ]first|work from home|wfh|100% remote|remote role|remote position)\b", t):
        return "remote"
    if re.search(r"\b(on[- ]?site|work from office|wfo|in[- ]office)\b", t):
        return "onsite"
    return None


def qualification(sents):
    for s in sents:
        if s["section"] in ("about", "benefits", "legal"):
            continue
        if _DEGREE_LINE.search(s["text"]) and matching.degree_levels(s["text"]):
            lvl = min(matching.degree_levels(s["text"]))           # "Bachelor's or Master's" asks for at least a bachelor's
            return s["text"][:200], matching.LEVEL_SHORT[lvl]
    return None, None


def structure_job(job, sents, skills, soft, discovered):
    text = "\n".join(s["text"] for s in sents)
    lo, hi = postingread._years(text)
    qual, level = qualification(sents)
    return {
        "title": job["title"], "company": job.get("company"), "location": job.get("location"),
        "employment_type": job.get("employment_type") or (re.search(r"\b(full[- ]time|part[- ]time|contract|internship)\b", text, re.I) or [None])[0],
        "work_mode": work_mode(f"{job['title']} {job.get('location') or ''} {text}"),
        "experience_min": lo, "experience_max": hi, "qualification": qual, "qualification_level": level,
        # tools count at full weight; domain phrases (banking, fraud detection) only as nice-to-have, matched by meaning
        "skills_required": [s["skill"] for s in skills if s["group"] == "tool" and s["need"] == "required"],
        "skills_nice": [s["skill"] for s in skills if s["group"] == "tool" and s["need"] == "nice"],
        "domain": [s["skill"] for s in skills if s["group"] == "domain"],
        "soft_skills": [s["soft_skill"] for s in soft],
        "responsibilities": [s["text"] for s in sents if s["section"] == "responsibilities"][:12],
        "discovered": discovered[:20],
        "parsed_by": "jobbert-2" if not DRY_RUN else "standin",
    }


JD = structure_job(JOB, SENTS, SKILLS, SOFT, DISCOVERED)
print(json.dumps({k: v for k, v in JD.items() if k != "responsibilities"}, indent=2, ensure_ascii=False))

{
  "title": "Senior Analyst, Digital Analytics",
  "company": "Labcorp",
  "location": "Bengaluru, Karnataka",
  "employment_type": "Full–time",
  "work_mode": null,
  "experience_min": 4,
  "experience_max": 6,
  "qualification": null,
  "qualification_level": null,
  "skills_required": [
    "power bi",
    "marketo",
    "powerpoint",
    "adobe journey optimizer",
    "adobe customer journey analytics"
  ],
  "skills_nice": [],
  "domain": [
    "website traffic",
    "data storytelling",
    "website analytics",
    "digital marketing",
    "customer journeys",
    "workspace reporting",
    "adobe cja certification",
    "digital analytics reporting"
  ],
  "soft_skills": [
    "data storytelling skills",
    "analytical and problem-solving skills",
    "data storytelling",
    "presentation skills",
    "business mindset"
  ],
  "discovered": [],
  "parsed_by": "jobbert-2"
}


## Step 8 · Read the resume the same way

- **Text:** Hiriq's own resume reader (PDF, DOCX or TXT).
- **Skills:** JobBERT's knowledge model over every line (it was trained on job posts, but tools and subjects read the same in a resume), plus any **Skills:** lines split on commas.
- **Years:** every date range in the resume ("Apr 2024 – Jun 2026", "06/2022 - Present"), with overlapping jobs counted once. Set `RESUME_YEARS` to override.
- **Degree:** the highest level named (Hiriq's degree reader).

In [86]:
from app.services import resume_text

_RANGE = re.compile(r"((?:[A-Za-z]{3,9}\.?,?\s+)?\d{4}|\d{1,2}[/.-]\d{4})\s*(?:-|–|—|to|till|until)\s*"
                    r"(present|current|now|till date|date|(?:[A-Za-z]{3,9}\.?,?\s+)?\d{4}|\d{1,2}[/.-]\d{4})", re.I)


def resume_years(text, today=None):
    today = today or date.today()
    spans = []
    for m in _RANGE.finditer(text):
        a, b = parse_month(m.group(1), today), parse_month(m.group(2), today, end=True)
        if a is not None and b is not None and a <= b:
            spans.append((a, b))
    if not spans:
        return None
    months, cur = 0, None                       # union of the ranges, so overlapping jobs count once
    for a, b in sorted(spans):
        if cur is None or a > cur[1] + 1:
            if cur:
                months += cur[1] - cur[0] + 1
            cur = [a, b]
        else:
            cur[1] = max(cur[1], b)
    months += cur[1] - cur[0] + 1
    return round(months / 12, 1)


def read_resume(path):
    data = Path(path).read_bytes()
    if not hasattr(resume_text, "looks_glued"):
        print("⚠ Your backend is older than the PDF fix: copy backend/app/services/resume_text.py from this update.")
    text = resume_text.extract_text(data)
    if hasattr(resume_text, "looks_glued") and resume_text.looks_glued(text):
        print("⚠ The resume text still has words run together. Check the PDF (send it to me if it keeps happening).")
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    found = {}
    for line, spans in zip(lines, tag([l[:1500] for l in lines], "knowledge")):
        for sp in spans:
            for part in pieces(sp["text"]):
                kind = classify(part, sp["score"])
                if kind == "known":
                    for k in known_inside(part):
                        found.setdefault(matching.canon(k), line)
                elif kind == "tool":                   # resumes keep only real tool names; domain words stay in the text
                    found.setdefault(matching.canon(part), line)
    for line in lines:                           # "Skills: SQL, Python, Power BI" lines
        m = re.match(r"^(?:•\s*)?(?:technical\s+)?(?:skills|tools|technologies|tech stack)\s*[:\-–]\s*(.+)$", line, re.I)
        if m:
            for part in _SPLIT.split(m.group(1)):
                if part.strip():
                    found.setdefault(matching.canon(part.strip()), line)
    levels = matching.degree_levels(text)
    return {
        "text": text, "lines": lines, "skills": sorted(found), "evidence": found,
        "years": RESUME_YEARS if RESUME_YEARS is not None else resume_years(text),
        "qualification": matching.LEVEL_SHORT[max(levels)] if levels else None,
        "education": [l for l in lines if matching.degree_levels(l)][:4],
    }


CV = read_resume(RESUME_PATH)
print(f"{len(CV['skills'])} skills · {CV['years']} years · highest degree: {CV['qualification']}")
print("first lines as read:", CV["lines"][:4])
print(CV["skills"])

⚠ Your backend is older than the PDF fix: copy backend/app/services/resume_text.py from this update.
76 skills · 2.9 years · highest degree: master's
first lines as read: ['Zahid Salim Shaikh', 'Mumbai,India | zahidshaikh10101@gmail.com | +91-8286092787 | Portfolio | LinkedIn | Github', 'M.TechinDataScience&EngineeringfromBITSPilaniwith4+yearsofexperiencebuildingscalabledata', 'productsandanalyticssystemsthatdrivemeasurablebusinessimpact. ExpertinPython,SQL,andPySpark-']
['a/b testing', 'afull-stackaicareerintelligenceplatformusingfastapiandopenrouterllmstoparse', 'agile', 'atr', 'bidashboards', 'borderline', 'builthirebot', 'businessintelligence', 'cdpracticeswithinadistributedteam', 'clickup', 'cohortanalysis', 'css', 'data-drivena', 'dataautomation', 'databricks', 'datacleaning', 'datamodelling', 'datapipeline', 'etlautomation', 'excel', 'exploratory data analysis', 'fastapi', 'featureengineering', 'generativeai', 'github', 'googleforjobs apiforlivejobdiscovery', 'huggingface', 'int

## Step 9 · Semantic match for what the rules miss

Hiriq's rules already match aliases (PowerBI = Power BI) and families (Tableau is *related* to Power BI). Embeddings catch what they can't: *Looker Studio* against *Google Data Studio*, *LLM APIs* against *OpenAI API integration*.

`all-MiniLM-L6-v2` turns each skill and each resume line into a vector, and we take the cosine similarity:

| Similarity | Verdict | Credit |
|---|---|---|
| rules say listed / in text / implied | match | 1.0 |
| rules say same family | related | 0.5 |
| embedding ≥ `RELATED_SIM` with a resume line | related (AI) | 0.5, with that line as the quote |
| below | missing | 0 |

Soft skills are not part of the %. They get their own coverage number against resume lines (`SOFT_SIM`).

In [87]:
def semantic_verdicts(jd, cv, missing):
    """For skills the rules call missing: the most similar resume line, if similar enough -> {canon: {"verdict": "related", "quote"}}."""
    lines = [l for l in cv["lines"] if len(l.split()) >= 2] + cv["skills"]
    sims = similarity(missing, lines)
    out, table = {}, []
    for skill, row in zip(missing, sims):
        if not row:
            continue
        j = max(range(len(row)), key=row.__getitem__)
        best, line = row[j], lines[j]
        quote = line if line in cv["text"] else cv["evidence"].get(line)       # always a real line of the resume
        table.append({"skill": skill, "best resume line": line[:100], "similarity": round(best, 3), "counts": best >= RELATED_SIM})
        if best >= RELATED_SIM and quote and matching.verify_quote(quote, cv["text"]):
            out[matching.canon(skill)] = {"verdict": "related", "quote": quote}
    return out, pd.DataFrame(table)


def soft_coverage(jd, cv):
    if not jd["soft_skills"]:
        return None, pd.DataFrame()
    sims = similarity(jd["soft_skills"], cv["lines"])
    rows = [{"soft skill": s, "best resume line": cv["lines"][max(range(len(r)), key=r.__getitem__)][:100], "similarity": round(max(r), 3)}
            for s, r in zip(jd["soft_skills"], sims) if r]
    df = pd.DataFrame(rows)
    return round(100 * (df["similarity"] >= SOFT_SIM).mean(), 1), df

## Step 10 · The match %

Hiriq's own scorer (`matching.score_candidate`) with the candidate-side weights: skills 42, experience 32, qualification 21, keywords 5. Location is off on the candidate side. Each required tool counts 1; each nice-to-have tool and each domain phrase counts 0.4. A part that can't be judged (no range, no dates) is left out and the rest re-weighted, never guessed.

In [88]:
def job_view(jd, job):
    return {"skills_required": jd["skills_required"], "skills_nice": jd["skills_nice"] + jd["domain"], "qualification": jd["qualification"],
            "experience_min": jd["experience_min"], "experience_max": jd["experience_max"], "location": job.get("location"),
            "work_mode": jd["work_mode"], "text": "\n".join(x for x in (job["title"], job["description"]) if x)}


def cand_view(cv):
    return {"skills": cv["skills"], "years": cv["years"], "location": None, "text": cv["text"],
            "qualification": cv["qualification"], "education": cv["education"]}


def missing_of(result):
    sk = result["explanation"]["skills"]
    return [r["skill"] for r in sk["required"] + sk["nice"] if r["status"] == "missing"]


def match(jd, job, cv, semantic=True):
    jv, cvv = job_view(jd, job), cand_view(cv)
    rules = matching.score_candidate(jv, cvv, CANDIDATE_WEIGHTS)
    if not semantic:
        return rules, {}, pd.DataFrame()
    verdicts, table = semantic_verdicts(jd, cv, missing_of(rules))
    return matching.score_candidate(jv, cvv, CANDIDATE_WEIGHTS, verdicts), verdicts, table


RESULT, VERDICTS, SEM_TABLE = match(JD, JOB, CV)
SOFT_PCT, SOFT_TABLE = soft_coverage(JD, CV)

print(f"MATCH: {RESULT['score']}%   (parts judged: {RESULT['assessed_weight']}% of the weight)")
print(RESULT["explanation"]["summary"])
display(pd.DataFrame([{"part": k, **{kk: v[kk] for kk in ("score", "weight", "assessed", "state")}} for k, v in RESULT["breakdown"]["parts"].items()]))

sk = RESULT["explanation"]["skills"]
display(pd.DataFrame([{"need": n, "skill": r["skill"], "status": r["status"], "credit": r["credit"], "evidence": (r["evidence"] or "")[:90]}
                      for n in ("required", "nice") for r in sk[n]]))
print("semantic check on the missing skills:")
display(SEM_TABLE)
print(f"soft skills covered: {SOFT_PCT}%")
display(SOFT_TABLE)

MATCH: 36.9%   (parts judged: 79% of the weight)
1 of 5 required skills found. Missing: marketo, powerpoint, adobe journey optimizer, adobe customer journey analytics. The job lists no qualification. 2.9 years against a minimum of 4 years. 4 of 25 key terms from the job description also appear in the resume. This score is based on 79% of the criteria; the rest could not be assessed.


,part,score,weight,assessed,state
0,skills,12.2,42,True,assessed
1,qualification,NaN,21,False,not_assessed
2,experience,72.5,32,True,below
3,location,NaN,0,False,unknown
4,keywords,16.0,5,True,assessed


,need,skill,status,credit,evidence
0,required,power bi,listed,1.0,"Visualisation&BI:PowerBI,Seaborn,Plotly,Matplotlib,Streamlit,Excel,Looker"
1,required,marketo,missing,0.0,
2,required,powerpoint,missing,0.0,
3,required,adobe journey optimizer,missing,0.0,
4,required,adobe customer journey analytics,missing,0.0,
5,nice,website traffic,missing,0.0,
6,nice,data storytelling,missing,0.0,
7,nice,website analytics,missing,0.0,
8,nice,digital marketing,missing,0.0,
9,nice,customer journeys,missing,0.0,


semantic check on the missing skills:


,skill,best resume line,similarity,counts
0,marketo,fastapi,0.335,False
1,powerpoint,reportlab,0.373,False
2,adobe journey optimizer,"• ArchitectedautomateddataingestionpipelinesinPythonintegrating4streamingDSPAPIs(Spotify,YouTube,",0.301,False
3,adobe customer journey analytics,dataautomation,0.351,False
4,website traffic,"WebDevelopmentIntern,VorfreudeCommunity–Mumbai Mar2020–May2020",0.369,False
5,data storytelling,dataautomation,0.600,False
6,website analytics,statistical,0.469,False
7,digital marketing,businessintelligence,0.426,False
8,customer journeys,businessintelligence,0.284,False
9,workspace reporting,reportlab,0.467,False


soft skills covered: 40.0%


,soft skill,best resume line,similarity
0,data storytelling skills,Skills,0.539
1,analytical and problem-solving skills,Skills,0.575
2,data storytelling,"performancetracking,data-drivenA&Rinvestmentdecisionsforleadership,andreducingper-artistinsight",0.382
3,presentation skills,Skills,0.555
4,business mindset,"StakeholderManagement,BusinessIntelligence",0.477


## Step 11 · JobBERT vs Hiriq's current reader

Same resume, same scorer. Only the way the job's skills are read changes.

In [89]:
def compare(job, cv):
    sents = sentences(job["description"])
    jd = structure_job(job, sents, *to_skills(tag_sentences(sents), sents)[:3])
    old = postingread.read(job["title"], job["description"])
    old_view = postingread.view(job["title"], job["description"], job.get("location"), old)
    old_res = matching.score_candidate(old_view, cand_view(cv), CANDIDATE_WEIGHTS)
    new_res, _, _ = match(jd, job, cv)
    return {
        "title": job["title"][:45], "company": job.get("company"),
        "rules skills": len(old["skills_required"]) + len(old["skills_nice"]),
        "JobBERT tools": len(jd["skills_required"]) + len(jd["skills_nice"]), "JobBERT domain": len(jd["domain"]),
        "only JobBERT found": sorted(set(jd["skills_required"] + jd["skills_nice"]) - set(old["skills_required"] + old["skills_nice"]))[:12],
        "years (rules / JobBERT step)": f"{old['experience_min']}-{old['experience_max']} / {jd['experience_min']}-{jd['experience_max']}",
        "rules %": old_res["score"], "JobBERT %": new_res["score"],
    }


display(pd.DataFrame([compare(j, CV) for j in JOBS]))

,title,company,rules skills,JobBERT tools,JobBERT domain,only JobBERT found,years (rules / JobBERT step),rules %,JobBERT %
0,Senior Data Analyst - AI & Automation,QualityAI,8,14,8,"[google apps script, google sheets, google slides, model context protocol, powerpoint, workspace]",5-8 / 5-8,67.4,46.5
1,"Senior Analyst, Digital Analytics",Labcorp,1,5,8,"[adobe customer journey analytics, adobe journey optimizer, marketo, powerpoint]",4-6 / 4-6,84.1,36.9


## Step 12 · Every scraped job, ranked for this resume

Point `JD_SOURCE` at a `gjobs_*.json` file from the scraper notebook and run this cell. It saves the structured jobs so we can check them against hand labels later (precision and recall per field).

In [90]:
def rank_all(jobs, cv):
    rows, structured = [], []
    for job in jobs:
        if not job.get("description"):
            continue
        sents = sentences(job["description"])
        jd = structure_job(job, sents, *to_skills(tag_sentences(sents), sents)[:3])
        res, _, _ = match(jd, job, cv)
        structured.append({**jd, "apply_url": job.get("apply_url"), "match": res["score"]})
        rows.append({"match %": res["score"], "title": job["title"][:50], "company": job.get("company"), "location": job.get("location"),
                     "years": f"{jd['experience_min']}-{jd['experience_max']}", "missing": missing_of(res)[:6], "apply_url": job.get("apply_url")})
    return pd.DataFrame(rows).sort_values("match %", ascending=False), structured


RANKED, STRUCTURED = rank_all(JOBS, CV)
display(RANKED)
Path("jobbert_structured.json").write_text(json.dumps(STRUCTURED, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved jobbert_structured.json")

,match %,title,company,location,years,missing,apply_url
0,46.5,Senior Data Analyst - AI & Automation,QualityAI,"Bengaluru, Karnataka",5-8,"[etl, workspace, powerpoint, google slides, google apps script, model context protocol]",https://in.linkedin.com/jobs/view/senior-data-analyst-ai-automation-at-qualityai-4484985251
1,36.9,"Senior Analyst, Digital Analytics",Labcorp,"Bengaluru, Karnataka",4-6,"[marketo, powerpoint, adobe journey optimizer, adobe customer journey analytics, website traffic, data storytelling]",https://careers.labcorp.com/senior-analyst-digital-analytics/job/2636637


saved jobbert_structured.json


## Step 13 · Accuracy against your hand labels

This step decides whether JobBERT gets merged. Use 10 posts of different kinds (BI, data science, operations research, a fresher post, one with few tools named).

1. Run 13a. It writes `labels.json`, pre-filled with what JobBERT found for every job in `JOBS`.
2. Open `labels.json` and **correct it by hand**: delete wrong skills, add missed ones, fix the years and degree level. Set `"checked": true` on each job you finish.
3. Run 13b. It scores JobBERT and Hiriq's current rules reader against your labels.

A predicted skill counts as correct when Hiriq treats it as the same skill as a labelled one ("PowerBI" = "Power BI"), or one contains the other ("ml/dl models" and "ml/dl"). **Merge when:** JobBERT's skill precision is ≥ 0.80, its recall is at least 15 points above the rules, and years and degree are as good as the rules.

| Metric | Meaning |
|---|---|
| precision | of the skills it returned, how many are real |
| recall | of the real skills, how many it found |
| F1 | the balance of the two |

In [91]:
LABELS_PATH = Path("labels.json")


def make_label_template(jobs, path=LABELS_PATH):
    old = {}
    if path.exists():
        old = {x["key"]: x for x in json.loads(path.read_text(encoding="utf-8"))}
    out = []
    for job in jobs:
        key = f"{job['title']} | {job.get('company')}"
        if key in old and old[key].get("checked"):
            out.append(old[key])                 # never overwrite your corrections
            continue
        sents = sentences(job["description"])
        jd = structure_job(job, sents, *to_skills(tag_sentences(sents), sents)[:3])
        out.append({"key": key, "checked": False, "skills": jd["skills_required"] + jd["skills_nice"] + jd["domain"],
                    "experience_min": jd["experience_min"], "experience_max": jd["experience_max"], "qualification_level": jd["qualification_level"],
                    "description": job["description"]})
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"wrote {path} with {len(out)} job(s); {sum(x['checked'] for x in out)} already checked")


make_label_template(JOBS)

wrote labels.json with 2 job(s); 0 already checked


In [92]:
def _same(a, b):
    ca, cb = matching.canon(a), matching.canon(b)
    return ca == cb or re.search(rf"\b{re.escape(ca)}\b", cb) is not None or re.search(rf"\b{re.escape(cb)}\b", ca) is not None


def prf(pred, gold):
    tp_p = sum(1 for p in pred if any(_same(p, g) for g in gold))
    tp_g = sum(1 for g in gold if any(_same(p, g) for p in pred))
    p = tp_p / len(pred) if pred else (1.0 if not gold else 0.0)
    r = tp_g / len(gold) if gold else 1.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def evaluate(path=LABELS_PATH):
    labels = [x for x in json.loads(path.read_text(encoding="utf-8")) if x.get("checked")]
    if not labels:
        print("No job in labels.json has \"checked\": true yet.")
        return None
    rows = []
    for lab in labels:
        title = lab["key"].split(" | ")[0]
        sents = sentences(lab["description"])
        jd = structure_job({"title": title, "description": lab["description"]}, sents, *to_skills(tag_sentences(sents), sents)[:3])
        old = postingread.read(title, lab["description"])
        for name, skills, lo, hi, lvl in (
            ("JobBERT", jd["skills_required"] + jd["skills_nice"] + jd["domain"], jd["experience_min"], jd["experience_max"], jd["qualification_level"]),
            ("rules", old["skills_required"] + old["skills_nice"], old["experience_min"], old["experience_max"],
             (matching.LEVEL_SHORT[min(matching.degree_levels(old["qualification"]))] if old["qualification"] and matching.degree_levels(old["qualification"]) else None)),
        ):
            p, r, f = prf(skills, lab["skills"])
            rows.append({"job": lab["key"][:40], "reader": name, "precision": round(p, 2), "recall": round(r, 2), "f1": round(f, 2),
                         "years ok": (lo, hi) == (lab["experience_min"], lab["experience_max"]), "degree ok": lvl == lab["qualification_level"],
                         "returned": len(skills), "labelled": len(lab["skills"])})
    df = pd.DataFrame(rows)
    display(df)
    summary = df.groupby("reader")[["precision", "recall", "f1", "years ok", "degree ok"]].mean().round(2)
    display(summary)
    jb, ru = summary.loc["JobBERT"], summary.loc["rules"]
    ready = jb["precision"] >= 0.80 and jb["recall"] - ru["recall"] >= 0.15 and jb["years ok"] >= ru["years ok"] and jb["degree ok"] >= ru["degree ok"]
    print(f"{len(labels)} labelled job(s). Merge gate:", "PASSED" if ready else "not yet",
          f"(precision {jb['precision']:.2f} ≥ 0.80, recall gain {jb['recall'] - ru['recall']:+.2f} ≥ +0.15)")
    return df


EVAL = evaluate()

No job in labels.json has "checked": true yet.
